# Heuristic Baselines Demo

This notebook demonstrates the heuristic link-prediction baseline pipeline on top of a saved split artifact.

It shows how to:

- load a deterministic split artifact
- inspect the train-only scoring graph
- score validation and test candidate edges with each heuristic
- build the shared experiment-result payload


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

PosixPath('/IdeaProjects/citegnn')

In [2]:
from citegnn.baselines.heuristics import HeuristicMethod, run_heuristic_baseline, score_split_with_heuristic
from citegnn.data.splits import load_split_artifact


In [3]:
dataset_name = "cora"
seed = 42
split_path = PROJECT_ROOT / "artifacts" / "splits" / f"{dataset_name}_seed{seed}_lp.pt"

if not split_path.exists():
    raise FileNotFoundError(
        f"Missing split artifact at {split_path}. Run scripts/build_link_prediction_split.py first."
    )

split = load_split_artifact(split_path)

{
    "dataset": split.dataset_name,
    "seed": split.seed,
    "is_undirected": split.is_undirected,
    "train_graph_edges": int(split.train_graph_edge_index.size(1)),
    "val_pos_edges": int(split.val_pos_edge_index.size(1)),
    "val_neg_edges": int(split.val_neg_edge_index.size(1)),
    "test_pos_edges": int(split.test_pos_edge_index.size(1)),
    "test_neg_edges": int(split.test_neg_edge_index.size(1)),
}

{'dataset': 'Cora',
 'seed': 42,
 'is_undirected': True,
 'train_graph_edges': 7392,
 'val_pos_edges': 527,
 'val_neg_edges': 527,
 'test_pos_edges': 1055,
 'test_neg_edges': 1055}

In [4]:
preview = {}

for method in HeuristicMethod:
    y_val_true, y_val_score, y_test_true, y_test_score = score_split_with_heuristic(split, method=method)
    preview[method.value] = {
        "val_examples": int(len(y_val_true)),
        "test_examples": int(len(y_test_true)),
        "val_score_min": float(y_val_score.min()),
        "val_score_max": float(y_val_score.max()),
        "test_score_min": float(y_test_score.min()),
        "test_score_max": float(y_test_score.max()),
    }

preview

{'common_neighbors': {'val_examples': 1054,
  'test_examples': 2110,
  'val_score_min': 0.0,
  'val_score_max': 6.0,
  'test_score_min': 0.0,
  'test_score_max': 9.0},
 'adamic_adar': {'val_examples': 1054,
  'test_examples': 2110,
  'val_score_min': 0.0,
  'val_score_max': 3.29211653140396,
  'test_score_min': 0.0,
  'test_score_max': 10.169186934627854},
 'personalized_pagerank': {'val_examples': 1054,
  'test_examples': 2110,
  'val_score_min': 0.0,
  'val_score_max': 0.23710450792732107,
  'test_score_min': 0.0,
  'test_score_max': 0.23710450792732107}}

In [5]:
results = [run_heuristic_baseline(split, method=method).to_dict() for method in HeuristicMethod]
results

[{'dataset': 'Cora',
  'method': 'common_neighbors',
  'seed': 42,
  'params': {'split_seed': 42,
   'is_undirected': True,
   'train_graph_edges': 7392},
  'validation': {'roc_auc': 0.6646155064829383,
   'average_precision': 0.6606916356330353,
   'precision': 0.5,
   'recall': 1.0,
   'f1': 0.6666666666666666,
   'threshold': 0.0,
   'positive_rate': 1.0},
  'test': {'roc_auc': 0.6716893151546461,
   'average_precision': 0.6708839892849784,
   'precision': 0.5,
   'recall': 1.0,
   'f1': 0.6666666666666666,
   'threshold': 0.0,
   'positive_rate': 1.0}},
 {'dataset': 'Cora',
  'method': 'adamic_adar',
  'seed': 42,
  'params': {'split_seed': 42,
   'is_undirected': True,
   'train_graph_edges': 7392},
  'validation': {'roc_auc': 0.6660665612881621,
   'average_precision': 0.6695342027783627,
   'precision': 0.5,
   'recall': 1.0,
   'f1': 0.6666666666666666,
   'threshold': 0.0,
   'positive_rate': 1.0},
  'test': {'roc_auc': 0.6716381033669504,
   'average_precision': 0.67053072851

To switch datasets, change `dataset_name` to `"pubmed"` and rerun the notebook.

For the CLI entrypoint, use `python scripts/run_heuristic_baselines.py --split-path artifacts/splits/cora_seed42_lp.pt`.